
# TT decomposition, superweight layers, and LoRA recovery experiments

**Llama-2-7B** `model.layers.1.mlp.down_proj`:

1. **Dense-reconstruction + LoRA**
   - decompose one superweight layer to TT,
   - reconstruct dense,
   - fine-tune only that layer with a LoRA adapter,
   - merge the LoRA adapter,
   - decompose and reconstruct the same layer again.

2. **Repeated decompose-reconstruct control**
   - decompose + reconstruct once,
   - then decompose + reconstruct the same layer again,
   - without LoRA.

3. **True TT-forward + LoRA**
   - decompose one superweight layer to `TTLinear`,
   - train a LoRA adapter on top of the frozen TT layer,
   - merge by reconstructing the TT layer to dense and adding the LoRA delta,
   - then decompose + reconstruct once more.


In [ ]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=6

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'src').exists() else CWD.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)
print('Has src:', (REPO_ROOT / 'src').exists())

In [ ]:
import gc
import json
import math
from dataclasses import dataclass

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.notebook import tqdm

from src.tt_llm import (
    TTLinear,
    cleanup_memory,
    factor_int_balanced,
    format_cuda_memory,
    get_module_by_name,
    infer_input_device,
    replace_llama_ffn_with_tt,
    replace_tt_with_dense_reconstruction,
    set_module_by_name,
)

try:
    from src.utils.eval import eval_ppl
except Exception:
    eval_ppl = None

pd.set_option('display.max_colwidth', 200)
torch.set_grad_enabled(True)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Configuration

In [ ]:
MODEL_NAME = 'meta-llama/Llama-2-7b-hf'
TARGET_LAYER = 1
TARGET_MODULE = 'mlp.down_proj'
TARGET_MODULE_NAME = f'model.layers.{TARGET_LAYER}.{TARGET_MODULE}'
TARGET_SUPERWEIGHT_COORDS = [(2533, 7890)]

TT_RANK = 500
ORDER = 12
TOKEN_CHUNK_SIZE = 128
DECOMPOSE_DEVICE = 'cpu'
DECOMPOSE_DTYPE = torch.float64
MODEL_DTYPE = torch.float16

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
MAX_STEPS = 100
GRAD_ACCUM_STEPS = 8
TRAIN_BATCH_SIZE = 1
TRAIN_SEQ_LEN = 256
NUM_TRAIN_SEQUENCES = 256

RUN_PPL = True
PPL_DATASETS = ['wikitext2']
PPL_SEQLEN = 4096

OUTPUT_JSON = REPO_ROOT / 'tt_lora_superweight_experiments.json'

## Helper functions

In [ ]:
def clean_memory_local(*objs):
    for obj in objs:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=MODEL_DTYPE,
        device_map='auto',
        low_cpu_mem_usage=True,
    )
    model.eval()
    if hasattr(model.config, 'use_cache'):
        model.config.use_cache = False
    return model, tokenizer


def build_wikitext2_train_loader(
    tokenizer,
    *,
    seq_len: int,
    batch_size: int,
    num_sequences: int,
    seed: int = 0,
):
    ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    text = "\n\n".join(ds["text"])
    encoded = tokenizer(text, return_tensors='pt')
    ids = encoded.input_ids[0]

    n_blocks = ids.numel() // seq_len
    ids = ids[: n_blocks * seq_len].view(n_blocks, seq_len)

    gen = torch.Generator().manual_seed(seed)
    if ids.shape[0] > num_sequences:
        perm = torch.randperm(ids.shape[0], generator=gen)[:num_sequences]
        ids = ids[perm]

    dataset = TensorDataset(ids)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


def evaluate_wikitext2_ppl(model, tokenizer, *, seqlen: int):
    if eval_ppl is None:
        raise RuntimeError('src.utils.eval.eval_ppl is not available in this repo layout')
    ppl = eval_ppl(model, tokenizer, datasets=['wikitext2'], seqlen=seqlen)
    return float(ppl['wikitext2'])


def get_target_linear(model):
    module = get_module_by_name(model, TARGET_MODULE_NAME)
    if not isinstance(module, (nn.Linear, TTLinear)):
        raise TypeError(f'Expected nn.Linear or TTLinear at {TARGET_MODULE_NAME}, got {type(module).__name__}')
    return module


def get_effective_target_weight(model) -> torch.Tensor:
    module = get_module_by_name(model, TARGET_MODULE_NAME)
    if isinstance(module, nn.Linear):
        return module.weight.detach().float().cpu().contiguous()
    if isinstance(module, TTLinear):
        return module.to_dense_weight().detach().float().cpu().contiguous()
    if hasattr(module, 'merged_linear'):
        merged = module.merged_linear()
        return merged.weight.detach().float().cpu().contiguous()
    raise TypeError(f'Unsupported target module type for dense-weight extraction: {type(module).__name__}')


def summarize_superweight_error(current_weight: torch.Tensor, baseline_weight: torch.Tensor, coords):
    rows = []
    for row, col in coords:
        orig = float(baseline_weight[int(row), int(col)].item())
        curr = float(current_weight[int(row), int(col)].item())
        abs_error = abs(curr - orig)
        rel_error = abs_error / max(abs(orig), 1e-12)
        rows.append({
            'row': int(row),
            'col': int(col),
            'original_value': orig,
            'current_value': curr,
            'abs_error': abs_error,
            'rel_error': rel_error,
        })
    detail_df = pd.DataFrame(rows)
    return {
        'superweight_abs_error_mean': float(detail_df['abs_error'].mean()),
        'superweight_abs_error_max': float(detail_df['abs_error'].max()),
        'superweight_rel_error_mean': float(detail_df['rel_error'].mean()),
        'superweight_rel_error_max': float(detail_df['rel_error'].max()),
        'superweight_error_detail': detail_df.to_dict(orient='records'),
    }


def decompose_target_layer_inplace(model, *, tt_rank: int):
    summaries = replace_llama_ffn_with_tt(
        model,
        layer_indices=[TARGET_LAYER],
        tt_rank=tt_rank,
        order=ORDER,
        projections=('down_proj',),
        decompose_dtype=DECOMPOSE_DTYPE,
        decompose_device=DECOMPOSE_DEVICE,
        token_chunk_size=TOKEN_CHUNK_SIZE,
    )
    return summaries


def reconstruct_target_layer_inplace(model):
    replace_tt_with_dense_reconstruction(model, [TARGET_MODULE_NAME])
    module = get_target_linear(model)
    if not isinstance(module, nn.Linear):
        raise TypeError('Expected reconstructed target layer to be nn.Linear')
    return module


def freeze_all_parameters(model):
    for p in model.parameters():
        p.requires_grad = False


def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
class LoRAOnFrozenModule(nn.Module):
    """
    Works on top of either nn.Linear or TTLinear.

    Forward:
        y = base(x) + scale * B(A(x))
    """
    def __init__(self, base_module: nn.Module, r: int = 16, alpha: int = 32, dropout: float = 0.0):
        super().__init__()
        if not isinstance(base_module, (nn.Linear, TTLinear)):
            raise TypeError(f'Unsupported base module: {type(base_module).__name__}')

        self.base_module = base_module
        self.in_features = int(base_module.in_features)
        self.out_features = int(base_module.out_features)
        self.r = int(r)
        self.alpha = int(alpha)
        self.scaling = float(alpha) / float(r)
        self.dropout = nn.Dropout(dropout)

        for p in self.base_module.parameters():
            p.requires_grad = False

        base_param = next(base_module.parameters())
        base_device = base_param.device
        
        self.lora_A = torch.nn.Parameter(
            torch.empty(r, self.in_features, device=base_device, dtype=torch.float32)
        )
        self.lora_B = torch.nn.Parameter(
            torch.zeros(self.out_features, r, device=base_device, dtype=torch.float32)
        )
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = self.base_module(x)
        x_work = self.dropout(x)
        x_lora = x_work.to(dtype=self.lora_A.dtype, device=self.lora_A.device)
        lora = (x_lora @ self.lora_A.t()) @ self.lora_B.t()
        lora = lora * self.scaling
        return base + lora.to(device=base.device, dtype=base.dtype)

    @torch.no_grad()
    def merged_linear(self) -> nn.Linear:
        if isinstance(self.base_module, TTLinear):
            base_weight = self.base_module.to_dense_weight().detach()
            base_bias = None if self.base_module.bias is None else self.base_module.bias.detach().clone()
            device = self.base_module.tt_cores[0].device
            dtype = self.base_module.tt_cores[0].dtype
        elif isinstance(self.base_module, nn.Linear):
            base_weight = self.base_module.weight.detach().clone()
            base_bias = None if self.base_module.bias is None else self.base_module.bias.detach().clone()
            device = self.base_module.weight.device
            dtype = self.base_module.weight.dtype
        else:
            raise TypeError(type(self.base_module).__name__)

        delta = (self.lora_B @ self.lora_A) * self.scaling
        delta = delta.to(device=device, dtype=dtype)
        merged = nn.Linear(
            self.in_features,
            self.out_features,
            bias=base_bias is not None,
            device=device,
            dtype=dtype,
        )
        merged.weight.copy_(base_weight.to(device=device, dtype=dtype) + delta)
        if base_bias is not None:
            merged.bias.copy_(base_bias.to(device=device, dtype=dtype))
        return merged


def attach_lora_to_target_module(model, *, r: int, alpha: int, dropout: float):
    base = get_target_linear(model)
    wrapper = LoRAOnFrozenModule(base, r=r, alpha=alpha, dropout=dropout)
    wrapper = wrapper.to(next(base.parameters()).device)
    set_module_by_name(model, TARGET_MODULE_NAME, wrapper)
    return wrapper


@torch.no_grad()
def merge_target_lora_to_dense_inplace(model):
    module = get_module_by_name(model, TARGET_MODULE_NAME)
    if not isinstance(module, LoRAOnFrozenModule):
        raise TypeError(f'Expected LoRAOnFrozenModule at target, got {type(module).__name__}')
    merged = module.merged_linear()
    set_module_by_name(model, TARGET_MODULE_NAME, merged)
    return merged

In [ ]:
def run_lora_training(
    model,
    tokenizer,
    *,
    max_steps: int,
    grad_accum_steps: int,
    batch_size: int,
    seq_len: int,
    num_sequences: int,
    lr: float,
    weight_decay: float,
    seed: int = 0,
):
    freeze_all_parameters(model)
    for name, p in model.named_parameters():
        if 'lora_' in name:
            p.requires_grad = True

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)
    loader = build_wikitext2_train_loader(
        tokenizer,
        seq_len=seq_len,
        batch_size=batch_size,
        num_sequences=num_sequences,
        seed=seed,
    )

    model.train()
    history = []
    device = infer_input_device(model)

    optimizer.zero_grad(set_to_none=True)
    step_count = 0
    tokens_seen = 0
    micro_step = 0
    pbar = tqdm(total=max_steps, desc='LoRA training', leave=True)

    loader_iter = iter(loader)

    while step_count < max_steps:
        try:
            (input_ids,) = next(loader_iter)
        except StopIteration:
            loader_iter = iter(loader)
            (input_ids,) = next(loader_iter)

        micro_step += 1
        input_ids = input_ids.to(device)
        tokens_seen += int(input_ids.numel())

        outputs = model(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss / grad_accum_steps
        loss.backward()

        if micro_step % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            step_count += 1
            current_loss = float(outputs.loss.detach().float().cpu().item())

            history.append(
                {
                    'step': step_count,
                    'tokens_seen': tokens_seen,
                    'loss': current_loss,
                }
            )

            pbar.update(1)
            pbar.set_postfix(loss=f'{current_loss:.4f}', tokens=tokens_seen)

    pbar.close()

    model.eval()
    return pd.DataFrame(history)


@torch.no_grad()
def stage_row(experiment: str, stage: str, model, tokenizer, baseline_target_weight: torch.Tensor):
    ppl = evaluate_wikitext2_ppl(model, tokenizer, seqlen=PPL_SEQLEN) if RUN_PPL else float('nan')
    module = get_module_by_name(model, TARGET_MODULE_NAME)
    current_weight = get_effective_target_weight(model)
    sw_stats = summarize_superweight_error(current_weight, baseline_target_weight, TARGET_SUPERWEIGHT_COORDS)
    return {
        'experiment': experiment,
        'stage': stage,
        'module_type': type(module).__name__,
        'wikitext2_ppl': ppl,
        **sw_stats,
    }

## Sanity check

In [ ]:
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
print(format_cuda_memory())
print('Target module:', TARGET_MODULE_NAME)
print('Target module type:', type(get_target_linear(model)).__name__)
print('Target module shape:', get_target_linear(model).out_features, 'x', get_target_linear(model).in_features)
clean_memory_local(model, tokenizer)

## Experiment 1 - decompose -> reconstruct -> LoRA -> merge -> decompose -> reconstruct

In [ ]:
def experiment_dense_reconstruct_lora():
    rows = []
    model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
    baseline_target_weight = get_effective_target_weight(model)

    rows.append(stage_row('exp1_dense_reconstruct_lora', 'baseline', model, tokenizer, baseline_target_weight))

    decompose_target_layer_inplace(model, tt_rank=TT_RANK)
    reconstruct_target_layer_inplace(model)
    rows.append(stage_row('exp1_dense_reconstruct_lora', 'after_first_decompose_reconstruct', model, tokenizer, baseline_target_weight))

    attach_lora_to_target_module(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
    print('Trainable parameters:', count_trainable_parameters(model))
    losses = run_lora_training(
        model,
        tokenizer,
        max_steps=MAX_STEPS,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        batch_size=TRAIN_BATCH_SIZE,
        seq_len=TRAIN_SEQ_LEN,
        num_sequences=NUM_TRAIN_SEQUENCES,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        seed=0,
    )
    rows.append(stage_row('exp1_dense_reconstruct_lora', 'after_lora_training', model, tokenizer, baseline_target_weight))

    merge_target_lora_to_dense_inplace(model)
    rows.append(stage_row('exp1_dense_reconstruct_lora', 'after_lora_merge', model, tokenizer, baseline_target_weight))

    decompose_target_layer_inplace(model, tt_rank=TT_RANK)
    reconstruct_target_layer_inplace(model)
    rows.append(stage_row('exp1_dense_reconstruct_lora', 'after_second_decompose_reconstruct', model, tokenizer, baseline_target_weight))

    out = pd.DataFrame(rows)
    clean_memory_local(model, tokenizer)
    return out, losses

In [ ]:
exp1_df, exp1_losses = experiment_dense_reconstruct_lora()
display(exp1_df)

In [ ]:
display(exp1_losses.head())

plt.figure(figsize=(7, 4.5))
plt.plot(exp1_losses['step'], exp1_losses['loss'], marker='o')
plt.xlabel('Optimization step')
plt.ylabel('Training loss')
plt.title('LoRA training loss vs step')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(exp1_losses['tokens_seen'], exp1_losses['loss'], marker='o')
plt.xlabel('Tokens seen')
plt.ylabel('Training loss')
plt.title('LoRA training loss vs tokens seen')
plt.grid(True)
plt.show()

## Experiment 2 - decompose -> reconstruct -> decompose -> reconstruct

In [ ]:
def experiment_repeated_decompose_reconstruct_control():
    rows = []
    model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
    baseline_target_weight = get_effective_target_weight(model)

    rows.append(stage_row('exp2_repeated_decompose_reconstruct', 'baseline', model, tokenizer, baseline_target_weight))

    decompose_target_layer_inplace(model, tt_rank=TT_RANK)
    reconstruct_target_layer_inplace(model)
    rows.append(stage_row('exp2_repeated_decompose_reconstruct', 'after_first_decompose_reconstruct', model, tokenizer, baseline_target_weight))

    decompose_target_layer_inplace(model, tt_rank=TT_RANK)
    reconstruct_target_layer_inplace(model)
    rows.append(stage_row('exp2_repeated_decompose_reconstruct', 'after_second_decompose_reconstruct', model, tokenizer, baseline_target_weight))

    out = pd.DataFrame(rows)
    clean_memory_local(model, tokenizer)
    return out

In [ ]:
exp2_df = experiment_repeated_decompose_reconstruct_control()
display(exp2_df)

## Experiment 3 - decompose -> LoRA with true TT-forward -> merge to dense -> decompose -> reconstruct

In [ ]:
def experiment_true_tt_forward_lora():
    rows = []
    model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
    baseline_target_weight = get_effective_target_weight(model)

    rows.append(stage_row('exp3_true_tt_lora', 'baseline', model, tokenizer, baseline_target_weight))

    decompose_target_layer_inplace(model, tt_rank=TT_RANK)
    rows.append(stage_row('exp3_true_tt_lora', 'after_tt_decompose_true_forward', model, tokenizer, baseline_target_weight))

    attach_lora_to_target_module(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
    print('Trainable parameters:', count_trainable_parameters(model))
    losses = run_lora_training(
        model,
        tokenizer,
        max_steps=MAX_STEPS,
        grad_accum_steps=GRAD_ACCUM_STEPS,
        batch_size=TRAIN_BATCH_SIZE,
        seq_len=TRAIN_SEQ_LEN,
        num_sequences=NUM_TRAIN_SEQUENCES,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        seed=1,
    )
    rows.append(stage_row('exp3_true_tt_lora', 'after_lora_training_on_tt_forward', model, tokenizer, baseline_target_weight))

    merge_target_lora_to_dense_inplace(model)
    rows.append(stage_row('exp3_true_tt_lora', 'after_lora_merge_to_dense', model, tokenizer, baseline_target_weight))

    decompose_target_layer_inplace(model, tt_rank=TT_RANK)
    reconstruct_target_layer_inplace(model)
    rows.append(stage_row('exp3_true_tt_lora', 'after_second_decompose_reconstruct', model, tokenizer, baseline_target_weight))

    out = pd.DataFrame(rows)
    clean_memory_local(model, tokenizer)
    return out, losses

In [ ]:
exp3_df, exp3_losses = experiment_true_tt_forward_lora()
display(exp3_df)

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(exp3_losses['step'], exp3_losses['loss'], marker='o')
plt.xlabel('Optimization step')
plt.ylabel('Training loss')
plt.title('TT-forward LoRA training loss vs step')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(exp3_losses['tokens_seen'], exp3_losses['loss'], marker='o')
plt.xlabel('Tokens seen')
plt.ylabel('Training loss')
plt.title('TT-forward LoRA training loss vs tokens seen')
plt.grid(True)
plt.show()

## Comparision

In [ ]:
all_results_df = pd.concat([exp2_df, exp3_df], ignore_index=True)
display(all_results_df)
all_results_df.to_json(OUTPUT_JSON, orient='records', indent=2)
print('Saved to', OUTPUT_JSON)

In [ ]:
plt.figure(figsize=(10, 5))
for experiment, group in all_results_df.groupby('experiment'):
    plt.plot(group['stage'], group['wikitext2_ppl'], marker='o', label=experiment)
plt.xticks(rotation=30, ha='right')
plt.ylabel('WikiText-2 PPL')
plt.title('PPL across TT/LoRA experiment stages')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
for experiment, group in all_results_df.groupby('experiment'):
    plt.plot(group['stage'], group['superweight_rel_error_mean'], marker='o', label=experiment)
plt.xticks(rotation=30, ha='right')
plt.ylabel('Mean superweight relative error')
plt.title('Superweight reconstruction error across TT/LoRA experiment stages')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
ppl_table = all_results_df.pivot(index='stage', columns='experiment', values='wikitext2_ppl')
display(ppl_table)

sw_rel_table = all_results_df.pivot(index='stage', columns='experiment', values='superweight_rel_error_mean')
display(sw_rel_table)

sw_abs_table = all_results_df.pivot(index='stage', columns='experiment', values='superweight_abs_error_mean')
display(sw_abs_table)

detail_cols = [
    'experiment',
    'stage',
    'module_type',
    'wikitext2_ppl',
    'superweight_abs_error_mean',
    'superweight_abs_error_max',
    'superweight_rel_error_mean',
    'superweight_rel_error_max',
]
display(all_results_df[detail_cols])